In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
!pip -q install scipy

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
from typing import Dict, Tuple

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# 텍스트 데이터 전처리
import re
# import contractions
from collections import Counter
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize

# 그래프 생성
import scipy.sparse as sp

# Tensorflow 라이브러리
import tensorflow as tf
tf.config.optimizer.set_jit(False)


# Tensorflow 모델 구조
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization,  Lambda, ReLU, Softmax
from tensorflow.keras.layers import Input, Embedding, Flatten, Layer, Multiply, Concatenate, Dropout, Dense, Reshape, Permute, Dot
from tensorflow.keras.utils import plot_model
from tensorflow.keras.initializers import GlorotUniform

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils


In [ ]:
def build_tf_sparse_adj(user_train, item_train, n_users, n_items):
    data = np.ones_like(user_train, dtype=np.float32)
    R_train = sp.coo_matrix((data, (user_train, item_train)), shape=(n_users, n_items))

    # 2) 무향 인접행렬 A 구성
    A = sp.bmat([[None,       R_train],
                 [R_train.T, None      ]], format='coo')

    # 3) Degree 행렬 D^-1/2 계산
    deg = np.array(A.sum(axis=1)).flatten()
    deg_inv_sqrt = np.power(deg, -0.5, where=deg>0)
    D_inv_sqrt = sp.diags(deg_inv_sqrt)

    # 4) 대칭 정규화: Ã = D^-1/2 · A · D^-1/2
    A_norm = D_inv_sqrt.dot(A).dot(D_inv_sqrt).tocoo()

    # 5) TensorFlow 희소텐서로 변환 (PyTorch 대신)
    indices = np.stack([A_norm.row.astype(np.int64),
                        A_norm.col.astype(np.int64)], axis=1)
    values = A_norm.data.astype(np.float32)
    shape = A_norm.shape
    adj_norm_tf = tf.SparseTensor(indices=indices, values=values, dense_shape=shape)
    adj_norm_tf = tf.sparse.reorder(adj_norm_tf)
    return adj_norm_tf

In [ ]:
def lightgcn_propagate(P0, adj_norm_tf, K):
    """
    P0: [|U|+|I|, d] 초기 ID 임베딩 (trainable)
    adj_norm_tf: tf.SparseTensor of Ã (size (|U|+|I|) x (|U|+|I|))
    return: P (all-layer average, Eq.(3)), P_list (per layer)
    """
    P_list = [P0]
    Pk = P0
    for _ in range(K):
        # Sparse x Dense
        Pk = tf.sparse.sparse_dense_matmul(adj_norm_tf, Pk)
        P_list.append(Pk)

    # Eq.(3) 균등 가중 평균
    P = tf.add_n(P_list) / float(K + 1)
    return P, P_list

In [ ]:
def build_modal_projectors(d_text_raw, d_vis_raw, d):

    # text embedding
    proj_text = Sequential([Input(shape=(d_text_raw,)), Dense(d, use_bias=True)], name="proj_text")

    # visual embedding
    proj_vis = Sequential([Input(shape=(d_vis_raw,)), Dense(d, use_bias=True)], name="proj_vis")

    return proj_text, proj_vis

In [ ]:
def self_cross_attention_block(num_heads, d, name_prefix="mm"):
    """
    Self-Attn: ID-Item(π_i) & modality(p_m) 쌍을 [ (pi,pv), (pv,pi), (pi,pi), (pv,pv) ] 순열로 구성 후
    single-layer MultiHeadAttention + Residual + LayerNorm + FFN (Eq.(6)-(9))
    Cross-Attn: 텍스트/비전 상호 정렬 (Eq.(10)-(13))
    """
    mha_self = MultiHeadAttention(num_heads=num_heads, key_dim=d, name=f"{name_prefix}_self_mha")
    mha_cross_vq_tkv = MultiHeadAttention(num_heads=num_heads, key_dim=d, name=f"{name_prefix}_cross_v_q_t_kv")
    mha_cross_tq_vkv = MultiHeadAttention(num_heads=num_heads, key_dim=d, name=f"{name_prefix}_cross_t_q_vd_kv")

    ln1 = LayerNormalization(name=f"{name_prefix}_ln1")
    ln2 = LayerNormalization(name=f"{name_prefix}_ln2")
    ln3 = LayerNormalization(name=f"{name_prefix}_ln3")
    ln4 = LayerNormalization(name=f"{name_prefix}_ln4")

    ffn1 = Sequential([Dense(2*d, activation='relu'), Dense(2*d)], name=f"{name_prefix}_ffn1")
    ffn2 = Sequential([Dense(2*d, activation='relu'), Dense(2*d)], name=f"{name_prefix}_ffn2")

    def block(pi, pv, pt):
        # pi: [B, d], pv: [B, d], pt: [B, d]
        # 시퀀스 구성 (Eq.(5)), feature dim = 2d
        def permute_seq(p_mod):
            # p_mod는 하나의 modality 벡터, 여기선 pv 또는 pt
            a = tf.concat([pi, p_mod], axis=-1)
            b = tf.concat([p_mod, pi], axis=-1)
            c = tf.concat([pi, pi], axis=-1)
            d2 = tf.concat([p_mod, p_mod], axis=-1)
            seq = tf.stack([a, b, c, d2], axis=1)  # [B, 4, 2d]
            return seq

        Hv_i = permute_seq(pv)  # [B, 4, 2d]
        Ht_i = permute_seq(pt)  # [B, 4, 2d]

        # Self-Attention (한 층만, 단순화)
        # query=key=value = Hv_i (또는 Ht_i)
        Hv_self = mha_self(Hv_i, Hv_i)
        Hv_self = ln1(Hv_i + Hv_self)
        Hv_self = ln2(Hv_self + ffn1(Hv_self))  # [B, 4, 2d]

        Ht_self = mha_self(Ht_i, Ht_i)
        Ht_self = ln1(Ht_i + Ht_self)
        Ht_self = ln2(Ht_self + ffn1(Ht_self))  # [B, 4, 2d]

        # Cross-Attention (텍스트<->비전 교차, 한 층만)
        # Query from one, Key/Value from the other (Eq.(10)-(13))
        Ht_to_v = mha_cross_vq_tkv(Hv_self, Ht_self, Ht_self)  # ΔH^c_{t→v}
        Hv_cross = ln3(Hv_self + Ht_to_v)
        Hv_cross = ln4(Hv_cross + ffn2(Hv_cross))  # [B, 4, 2d]

        Hv_to_t = mha_cross_tq_vkv(Ht_self, Hv_self, Hv_self)  # ΔH^c_{v→t}
        Ht_cross = ln3(Ht_self + Hv_to_t)
        Ht_cross = ln4(Ht_cross + ffn2(Ht_cross))  # [B, 4, 2d]

        # Global Max Pooling → 2d (논문 Multi-Modal Fusion 입력)
        gv = tf.reduce_max(Hv_cross, axis=1)  # [B, 2d]
        gt = tf.reduce_max(Ht_cross, axis=1)  # [B, 2d]

        return gv, gt

    return block

In [ ]:
def fusion_mlp(d, name="fusion_mlp"):
    # Concat(gv, gt, gv ⊙ gt) → MLP → h_i (Eq.(14))
    inp = Input(shape=(4*d + 2*d,))  # 2d + 2d + 2d = 6d = (4d + 2d) for clarity
    x = Dense(2*d, activation='relu')(inp)
    x = Dense(2*d, activation='relu')(x)
    out = Dense(d)(x)
    return Model(inp, out, name=name)

# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_final.pkl')
print(df.shape)
df.head()

(66718, 17)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06857138, 0.13921358, -0.039587017, -0.088...","Based on user reviews, it appears that users a...",The image shows a hand holding a set of earrin...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.04617965, 0.12618569, -0.017711096, -0.102...","Based on user reviews, it appears that users p...",The image shows a close-up of a person's ear w...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.062135138, 0.1333665, -0.019405615, -0.102...","Based on this review, it seems that users pref...","The image shows a fashion accessory, specifica..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06780286, 0.051634714, -0.017799914, -0.11...","Based on user reviews, it appears that users p...",The image shows a sleeveless tank top with a p...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.07941287, 0.05176169, -0.02554227, -0.0903...","Based on the user review, it appears that the ...",The image shows a lime green t-shirt with a gr...


In [ ]:
df['user_id'].nunique()

7449

In [ ]:
df['asin'].nunique()

16099

## Create Item Embedding

In [ ]:
# asin(item)별 groupby 후 평균
item_emb_df = df.groupby('asin').agg({
    'roberta_emb': lambda x: np.mean(np.stack(x), axis=0),
    'VGG16_emb': lambda x: np.mean(np.stack(x), axis=0)
}).reset_index()

item_emb_df.columns = ['asin', 'item_text_emb', 'item_image_emb']

# 데이터셋에 결과를 concat
df = df.merge(item_emb_df, on='asin', how='left')

# 결과 확인
item_emb_df.head()

,asin,item_text_emb,item_image_emb
0,1737581736,"[-0.0709785552, 0.09040954275, -0.020697090466...","[0.0, 16.58531626066667, 0.0, 3.77627869448333..."
1,B00068TJIU,"[-0.10145816703333332, 0.11970292776666668, -0...","[0.0, 11.138919035599999, 0.0, 3.7927707036333..."
2,B00068TJM6,"[-0.08949773261111112, 0.10334104133333336, -0...","[0.0, 44.68173302543333, 1.1680200629777777, 7..."
3,B00068TK70,"[-0.07676338526666666, 0.09960119676666668, -0...","[0.0, 34.207046508766666, 0.35857764880000004,..."
4,B0006AAS5G,"[-0.11626560985, 0.107114367175, -0.0297976802...","[0.0, 27.227974176399997, 0.0, 6.60296964645, ..."


# User, Item label encoding
- 라벨 인코딩
- train, test 분리



In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

In [ ]:
# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

# train과 test data를 분리
train, test = train_test_split(df, shuffle=True, test_size=0.2, random_state=42)
train, valid = train_test_split(train, shuffle=True, test_size=0.125, random_state=42)

# index reset
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
# User ID
user_id_train = train['user_id'].to_numpy()
user_id_valid = valid['user_id'].to_numpy()
user_id_test = test['user_id'].to_numpy()

# Item ID
item_id_train = train['asin'].to_numpy()
item_id_valid = valid['asin'].to_numpy()
item_id_test = test['asin'].to_numpy()

# Image Set Mean by Item
image_feat_train = np.stack(train['VGG16_emb'].values).astype(np.float32)
image_feat_valid = np.stack(valid['VGG16_emb'].values).astype(np.float32)
image_feat_test = np.stack(test['VGG16_emb'].values).astype(np.float32)

# Text Feature Mean by Item
text_feat_train = np.stack(train['roberta_emb'].to_numpy()).astype(np.float32)
text_feat_valid = np.stack(valid['roberta_emb'].to_numpy()).astype(np.float32)
text_feat_test = np.stack(test['roberta_emb'].to_numpy()).astype(np.float32)

# rating
y_train = train['rating'].to_numpy()
y_valid = valid['rating'].to_numpy()
y_test = test['rating'].to_numpy()

In [ ]:
adj_norm_tf = build_tf_sparse_adj(user_id_train, item_id_train, n_users, n_items)

# MMCN

In [ ]:
class MMCN(tf.keras.Model):
    """
    B안: MSE(주손실) + β*(Feature/Element 대조 보조항) + L2 정규화
    - compile(loss=None)
    - train_step/test_step에서 손실 직접 계산
    """
    def __init__(self, n_users, n_items, d_text_raw, d_vis_raw, adj_norm_tf,
                 d=64, K=3, num_heads=2, dropout_r=0.5, lambda_a=0.1, lambda_r=0.1, tau=0.999, beta=0.1, **kwargs):
        super().__init__(**kwargs)
        self.n_users = n_users
        self.n_items = n_items
        self.N = n_users + n_items
        self.d = d
        self.K = K
        self.num_heads = num_heads
        self.dropout_r = dropout_r
        self.lambda_a = lambda_a
        self.lambda_r = lambda_r
        self.tau = tau
        self.beta = tf.constant(beta, dtype=tf.float32)

        # LightGCN 초기 ID 임베딩
        self.P0 = self.add_weight(
            shape=(self.N, d),
            initializer=tf.keras.initializers.GlorotUniform(),
            trainable=True, name="ID_emb"
        )

        # 모달 투영
        self.proj_text, self.proj_vis = build_modal_projectors(d_text_raw, d_vis_raw, d)

        # 멀티모달 블록
        self.mm_block = self_cross_attention_block(num_heads, d, name_prefix="mm")
        self.fuse = fusion_mlp(d)

        # Predictor (feature-level online)
        self.predictor = Dense(d, use_bias=True, name="predictor")

        # Element-wise projectors
        self.elem_proj_online = Sequential([
            Dense(2*d, activation='relu'),
            Dense(d*d)
        ], name="elem_proj_online")
        self.elem_proj_target = Sequential([
            Dense(2*d, activation='relu'),
            Dense(d*d)
        ], name="elem_proj_target")

        # EMA
        self._ema_initialized = False
        self.adj = adj_norm_tf

        # 입력 placeholder
        self.user_id_inp = Input(shape=(), dtype=tf.int32, name="user_id")
        self.item_id_inp = Input(shape=(), dtype=tf.int32, name="item_id")
        self.text_inp = Input(shape=(d_text_raw,), dtype=tf.float32, name="item_text")
        self.vision_inp = Input(shape=(d_vis_raw,), dtype=tf.float32, name="item_vision")

        # === Trackers (train) ===
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.mse_tracker  = tf.keras.metrics.Mean(name="mse")
        self.feat_tracker = tf.keras.metrics.Mean(name="L_feature")
        self.elem_tracker = tf.keras.metrics.Mean(name="L_element")
        self.reg_tracker  = tf.keras.metrics.Mean(name="L_reg")

        # === Trackers (val) ===
        self.val_loss_tracker = tf.keras.metrics.Mean(name="val_loss")
        self.val_mse_tracker  = tf.keras.metrics.Mean(name="val_mse")
        self.val_feat_tracker = tf.keras.metrics.Mean(name="val_L_feature")
        self.val_elem_tracker = tf.keras.metrics.Mean(name="val_L_element")
        self.val_reg_tracker  = tf.keras.metrics.Mean(name="val_L_reg")

    @property
    def metrics(self):
        # Keras가 epoch마다 reset_states()를 호출할 stateful metric 목록
        return [
            self.loss_tracker, self.mse_tracker, self.feat_tracker, self.elem_tracker, self.reg_tracker,
            self.val_loss_tracker, self.val_mse_tracker, self.val_feat_tracker, self.val_elem_tracker, self.val_reg_tracker
        ]

    # ---------------- EMA utils ----------------
    def build(self, input_shapes):
        self.predictor.build((None, self.d))
        self.elem_proj_online.build((None, self.d))
        online_vars = self._collect_online_vars()
        self.ema_vars = [
            self.add_weight(
                name=f"ema_{w.name.replace(':','_')}",
                shape=w.shape,
                initializer="zeros",
                trainable=False,
            ) for w in online_vars
        ]
        for slot, w in zip(self.ema_vars, online_vars):
            slot.assign(w)
        self._ema_initialized = True
        super().build(input_shapes)

    def _collect_online_vars(self):
        return [self.P0] + self.predictor.weights + self.elem_proj_online.weights

    def _ema_update(self):
        online_vars = self._collect_online_vars()
        for slot, v in zip(self.ema_vars, online_vars):
            slot.assign(self.tau * slot + (1. - self.tau) * v)

    def _ema_sync_to_target(self):
        offset = 1 + len(self.predictor.weights)  # [P0] + predictor
        ema_elem = self.ema_vars[offset: offset + len(self.elem_proj_online.weights)]
        for w_t, w_ema in zip(self.elem_proj_target.weights, ema_elem):
            w_t.assign(w_ema)

    def _ema_init_if_needed(self):
        if not self._ema_initialized:
            online_vars = self._collect_online_vars()
            self.ema_vars = [self.add_weight(shape=v.shape, initializer="zeros", trainable=False)
                             for v in online_vars]
            for slot, v in zip(self.ema_vars, online_vars):
                slot.assign(v)
            self._ema_initialized = True

    # ---------------- forward ----------------
    def call(self, inputs, training=False):
        u = tf.cast(inputs["user_id"], tf.int32)
        i = tf.cast(inputs["item_id"], tf.int32)
        t_raw = inputs["item_text"]
        v_raw = inputs["item_vision"]

        # LightGCN propagation
        P, _ = lightgcn_propagate(self.P0, self.adj, self.K)  # [N,d]
        Pu = tf.gather(P, u)                 # [B,d]
        Pi = tf.gather(P, self.n_users + i)  # [B,d]

        # Modal projections
        pt = self.proj_text(t_raw)  # [B,d]
        pv = self.proj_vis(v_raw)   # [B,d]

        # Multi-modal fuse -> h_i
        gv, gt = self.mm_block(Pi, pv, pt)       # [B,2d], [B,2d]
        gmul = gv * gt                            # [B,2d]
        hi = self.fuse(tf.concat([gv, gt, gmul], axis=-1))  # [B,d]

        # Residual
        Pi_m = Pi + hi  # [B,d]

        # Online feature-level predictor
        Pu_online  = self.predictor(Pu)
        Pi_online  = self.predictor(Pi)
        Pi_m_online= self.predictor(Pi_m)

        # Target feature-level with multiplicative dropout
        if training:
            mask_u = tf.nn.dropout(tf.ones_like(Pu), rate=self.dropout_r) * (1.0 - self.dropout_r)
            mask_i = tf.nn.dropout(tf.ones_like(Pi), rate=self.dropout_r) * (1.0 - self.dropout_r)
        else:
            mask_u = tf.ones_like(Pu); mask_i = tf.ones_like(Pi)
        Pu_target = Pu * mask_u
        Pi_target = Pi * mask_i

        # Element-wise projectors
        def proj2mat(layer, x): return tf.reshape(layer(x), [-1, self.d, self.d])
        E_u_online = proj2mat(self.elem_proj_online, Pu_online)
        E_i_online = proj2mat(self.elem_proj_online, Pi_online)
        E_im_online= proj2mat(self.elem_proj_online, Pi_m_online)
        E_u_target = proj2mat(self.elem_proj_target, Pu_target)
        E_i_target = proj2mat(self.elem_proj_target, Pi_target)

        # Score (논문 Eq.(26)): 온라인 뷰 내적
        score = tf.reduce_sum(Pu_online * Pi_m_online, axis=-1)  # [B]

        return {
            "score": score,
            "Pu_online": Pu_online, "Pi_online": Pi_online, "Pi_m_online": Pi_m_online,
            "Pu_target": Pu_target, "Pi_target": Pi_target,
            "E_u_online": E_u_online, "E_i_online": E_i_online, "E_im_online": E_im_online,
            "E_u_target": E_u_target, "E_i_target": E_i_target
        }

    # ---------------- losses ----------------
    @staticmethod
    def neg_cosine(a, b, eps=1e-9):
        a = tf.math.l2_normalize(a, axis=-1)
        b = tf.math.l2_normalize(b, axis=-1)
        return -tf.reduce_mean(tf.reduce_sum(a * b, axis=-1))

    def compute_contrastive_loss(self, outs):
        # Feature-level
        L_feature_rec = self.neg_cosine(outs["Pu_online"], tf.stop_gradient(outs["Pi_target"])) \
                      + self.neg_cosine(tf.stop_gradient(outs["Pu_target"]), outs["Pi_online"])
        L_feature_align = self.neg_cosine(tf.stop_gradient(outs["Pi_target"]), outs["Pi_m_online"])
        L_feature = L_feature_rec + self.lambda_a * L_feature_align
        # Element-wise (flatten)
        def flat(x): return tf.reshape(x, [tf.shape(x)[0], -1])
        L_element_rec = self.neg_cosine(flat(outs["E_u_online"]), tf.stop_gradient(flat(outs["E_i_target"]))) \
                      + self.neg_cosine(tf.stop_gradient(flat(outs["E_u_target"])), flat(outs["E_i_online"]))
        L_element_align = self.neg_cosine(tf.stop_gradient(flat(outs["E_i_target"])), flat(outs["E_im_online"]))
        L_element = L_element_rec + self.lambda_a * L_element_align
        return L_feature, L_element

    # ---------------- train/val steps ----------------
    @tf.function(reduce_retracing=True)
    def train_step(self, data):
        if isinstance(data, (tuple, list)):
            inputs, y = data
        else:
            inputs, y = data, None

        with tf.GradientTape() as tape:
            outs = self(inputs, training=True)
            self._ema_init_if_needed()

            # 보조 대조손실
            L_feature, L_element = self.compute_contrastive_loss(outs)

            # 정규화(배치 근사)
            L_reg = tf.reduce_mean(
                tf.reduce_sum(tf.square(outs["Pu_online"]), axis=-1) +
                tf.reduce_sum(tf.square(outs["Pi_online"]), axis=-1)
            )

            # === 주손실: MSE ===
            y_pred = outs["score"]
            if y is not None:
                y = tf.cast(y, y_pred.dtype)
                y_pred = tf.reshape(y_pred, tf.shape(y))
                mse = tf.reduce_mean(tf.square(y_pred - y))
            else:
                mse = tf.constant(0.0, dtype=y_pred.dtype)

            total_loss = mse + self.beta * (L_feature + L_element) + self.lambda_r * L_reg

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # EMA 업데이트 및 타깃 동기화
        self._ema_update()
        self._ema_sync_to_target()

        return {
            "loss": total_loss,
            "mse": mse,
            "L_feature": L_feature,
            "L_element": L_element,
            "L_reg": L_reg,
        }


    @tf.function(reduce_retracing=True)
    def test_step(self, data):
        if isinstance(data, (tuple, list)):
            inputs, y = data
        else:
            inputs, y = data, None

        outs = self(inputs, training=False)

        # === 손실 계산 (네 코드 그대로) ===
        L_feature, L_element = self.compute_contrastive_loss(outs)
        L_reg = tf.reduce_mean(
            tf.reduce_sum(tf.square(outs["Pu_online"]), axis=-1) +
            tf.reduce_sum(tf.square(outs["Pi_online"]), axis=-1)
        )
        y_pred = outs["score"]
        if y is not None:
            y = tf.cast(y, y_pred.dtype)
            y_pred = tf.reshape(y_pred, tf.shape(y))
            mse = tf.reduce_mean(tf.square(y_pred - y))
        else:
            mse = tf.constant(0.0, dtype=y_pred.dtype)
        total_loss = mse + self.beta * (L_feature + L_element) + self.lambda_r * L_reg

        # ✅ 여기서는 'val_' 없이 반환. Keras가 자동으로 'val_'를 붙임.
        return {
            "loss": total_loss,
            "mse": mse,
            "L_feature": L_feature,
            "L_element": L_element,
            "L_reg": L_reg,
        }

In [ ]:
# 하이퍼 파라미터
d = 64
K = 3
num_heads = 2
dropout_r = 0.3
lambda_a = 0.1
lambda_r = 0.01
tau = 0.999
lr = 1e-3
beta=1e-5

In [ ]:
mmcn = MMCN(
    n_users=n_users, n_items=n_items,
    d_text_raw=768, d_vis_raw=4096,
    adj_norm_tf=adj_norm_tf,
    d=d, K=K, num_heads=num_heads,
    dropout_r=dropout_r, lambda_a=lambda_a, lambda_r=lambda_r,
    tau=tau,
)

# 타깃 네트워크는 EMA로만 갱신
mmcn.elem_proj_online.trainable = True
mmcn.elem_proj_target.trainable = False

mmcn.compile(
    optimizer=tf.keras.optimizers.Adam(lr),
    run_eagerly=False,     # 디버그 편의
    jit_compile=False
)

In [ ]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.01,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [ ]:
train_inputs = {
    "user_id":    user_id_train.astype('int32'),      # int32, shape (N,)
    "item_id":    item_id_train.astype('int32'),      # int32, shape (N,)
    "item_text":  text_feat_train.astype('float32'),    # float32, shape (N, d_text_raw)
    "item_vision": image_feat_train.astype('float32'),  # float32, shape (N, d_vis_raw)
}
valid_inputs = {
    "user_id":    user_id_valid.astype('int32'),
    "item_id":    item_id_valid.astype('int32'),
    "item_text":  text_feat_valid.astype('float32'),
    "item_vision": image_feat_valid.astype('float32'),
}

history = mmcn.fit(
    x=train_inputs,
    y=y_train,
    validation_data=(valid_inputs, y_valid),
    epochs=100,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - L_element: -2.0913 - L_feature: -0.3171 - L_reg: 0.0923 - loss: 0.5388 - mse: 0.7787 - val_L_element: -2.0996 - val_L_feature: -0.5651 - val_L_reg: 0.1290 - val_loss: -0.0374 - val_mse: 0.2278
Epoch 2/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - L_element: -2.0996 - L_feature: -0.5393 - L_reg: 0.1366 - loss: -0.0340 - mse: 0.2285 - val_L_element: -2.0991 - val_L_feature: -0.7335 - val_L_reg: 0.1493 - val_loss: -0.0349 - val_mse: 0.2469
Epoch 3/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - L_element: -2.0991 - L_feature: -0.6417 - L_reg: 0.1575 - loss: -0.1040 - mse: 0.1685 - val_L_element: -2.0984 - val_L_feature: -0.8048 - val_L_reg: 0.1717 - val_loss: 0.0469 - val_mse: 0.3355
Epoch 4/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - L_element: -2.0984 - L_feature: -0.6838 - L_reg: 0.1769 - loss: -0.1440 - mse: 0.1324 - val_L_element: -2.0977 - val_L_feature: -0.8611 - val_L_reg: 0.1878 - val_loss: 0.0450 - val_mse: 0.3390
Ep

In [ ]:
# 1) 테스트 입력 준비 (dict)
test_inputs = {
    "user_id":    user_id_test.astype('int32'),
    "item_id":    item_id_test.astype('int32'),
    "item_text":  text_feat_test.astype('float32'),
    "item_vision": image_feat_test.astype('float32'),
}

# 2) 예측 (회귀 점수)
y_pred = mmcn.predict(test_inputs)
y_pred = np.asarray(y_pred['score'], dtype=np.float32).ravel()

# MAE
MAE = mean_absolute_error(y_test, y_pred)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, y_pred)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"MAPE: {MAPE:.4f}%")

417/417 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step
MAE: 0.3810
MSE: 0.3052
RMSE: 0.5524
MAPE: 10.9431%
